In [2]:
import os
import re
import pandas as pd
from sklearn.model_selection import train_test_split

# ----------------------------------------------------
# File Paths
# ----------------------------------------------------
INPUT_FILE = r"C:\Users\Dypiu 539-15\Downloads\Phishing URL dataset\Phishing URL dataset\dataset2.csv"
OUTPUT_DIR = r"C:\Users\Dypiu 539-15\Downloads\Phishing URL dataset\Phishing URL dataset"

TRAIN_FILE = os.path.join(OUTPUT_DIR, "train_dataset.csv")
TEST_FILE = os.path.join(OUTPUT_DIR, "test_dataset.csv")

# ----------------------------------------------------
# Load Dataset
# ----------------------------------------------------
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Could not find the dataset at {INPUT_FILE}")

df = pd.read_csv(INPUT_FILE)
print(f"Dataset Loaded Successfully. Initial Rows: {len(df)}")

# Create a mapping of lowercase column names to their actual names in the dataframe
col_mapping = {col.lower(): col for col in df.columns}

# Dynamically locate the URL column and the Label column (case-insensitive)
possible_url_cols = ["url", "urls", "link", "text", "domain"]
url_column = next((col_mapping[col] for col in possible_url_cols if col in col_mapping), None)

possible_label_cols = ["label", "class", "result", "status", "prediction"]
label_column = next((col_mapping[col] for col in possible_label_cols if col in col_mapping), None)

if not url_column or not label_column:
    raise Exception(f"Could not map columns. Found columns: {df.columns.tolist()}\n"
                    f"Please ensure your CSV has a URL-like column and a Label-like column.")

print(f"Using '{url_column}' as URL column and '{label_column}' as Target Label column.\n")

# ----------------------------------------------------
# Step 1: Remove Duplicate URLs
# ----------------------------------------------------
initial_count = len(df)
df = df.drop_duplicates(subset=[url_column])
print(f"[Step 1] Removed {initial_count - len(df)} duplicate records.")

# ----------------------------------------------------
# Step 2: Remove Missing Values
# ----------------------------------------------------
before_missing = len(df)
df = df.dropna(subset=[url_column, label_column])
print(f"[Step 2] Removed {before_missing - len(df)} rows with missing values.")

# ----------------------------------------------------
# Step 3: Convert Labels into Binary Form
# ----------------------------------------------------
df['binary_label'] = df[label_column].apply(lambda x: 1 if "phish" in str(x).lower() or str(x) in ["1", "bad", "yes"] else 0)
print(f"[Step 3] Labels converted to binary format.")
print(f"         Class Summary -> Phishing (1): {sum(df['binary_label'] == 1)} | Legitimate (0): {sum(df['binary_label'] == 0)}")

# ----------------------------------------------------
# Step 4 & 5: Normalize URLs & Remove Unnecessary Symbols
# ----------------------------------------------------
def normalize_and_clean_url(url):
    if not isinstance(url, str):
        return ""
    
    url_clean = url.lower()
    url_clean = re.sub(r"https?://", "", url_clean)
    url_clean = re.sub(r"^www\.", "", url_clean)
    
    # Split by common symbols
    url_clean = re.split(r'[/.\-_?=&]+', url_clean)
    
    return " ".join([word for word in url_clean if word.strip() != ""])

print("[Step 4 & 5] Normalizing URLs and stripping structural syntax symbols...")
df['normalized_url'] = df[url_column].apply(normalize_and_clean_url)

# Remove any rows that became completely empty strings after cleaning
df = df[df['normalized_url'] != ""]

# ----------------------------------------------------
# Step 6: Split Dataset (Training: 80%, Testing: 20%)
# ----------------------------------------------------
print("[Step 6] Partitioning dataset into 80% Training and 20% Testing chunks...")

if len(df['binary_label'].unique()) > 1:
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['binary_label'])
else:
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# ----------------------------------------------------
# Save Outputs
# ----------------------------------------------------
train_df.to_csv(TRAIN_FILE, index=False)
test_df.to_csv(TEST_FILE, index=False)

print("\n==================================================")
print("DATA PREPROCESSING COMPLETE!")
print(f"Total processed rows : {len(df)}")
print(f" -> Training Rows (80%): {len(train_df)}")
print(f" -> Testing Rows  (20%): {len(test_df)}")
print("\nFiles Saved At:")
print(f" - Train Split: {TRAIN_FILE}")
print(f" - Test Split : {TEST_FILE}")
print("==================================================")

Dataset Loaded Successfully. Initial Rows: 149726
Using 'URL' as URL column and 'Label' as Target Label column.

[Step 1] Removed 19949 duplicate records.
[Step 2] Removed 0 rows with missing values.
[Step 3] Labels converted to binary format.
         Class Summary -> Phishing (1): 54805 | Legitimate (0): 74972
[Step 4 & 5] Normalizing URLs and stripping structural syntax symbols...
[Step 6] Partitioning dataset into 80% Training and 20% Testing chunks...

DATA PREPROCESSING COMPLETE!
Total processed rows : 129777
 -> Training Rows (80%): 103821
 -> Testing Rows  (20%): 25956

Files Saved At:
 - Train Split: C:\Users\Dypiu 539-15\Downloads\Phishing URL dataset\Phishing URL dataset\train_dataset.csv
 - Test Split : C:\Users\Dypiu 539-15\Downloads\Phishing URL dataset\Phishing URL dataset\test_dataset.csv


In [3]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ----------------------------------------------------
# File Paths
# ----------------------------------------------------
# Reading the preprocessed training dataset generated in the previous step
INPUT_FILE = r"C:\Users\Dypiu 539-15\Downloads\Phishing URL dataset\Phishing URL dataset\train_dataset.csv"
OUTPUT_DIR = r"C:\Users\Dypiu 539-15\Downloads\Phishing URL dataset\Phishing URL dataset"
GRAPH_OUTPUT_FILE = os.path.join(OUTPUT_DIR, "dataset_plots.png")

# ----------------------------------------------------
# 1. Load Preprocessed Data
# ----------------------------------------------------
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Could not find the preprocessed training file at {INPUT_FILE}. Please run the preprocessing script first.")

df = pd.read_csv(INPUT_FILE)
print(f"Dataset Loaded Successfully: {len(df)} rows found for plotting.")

# ----------------------------------------------------
# 2. Prepare Labels for the Visual Graph
# ----------------------------------------------------
# Create a human-readable column for the chart axis labels
df['Class Name'] = df['binary_label'].map({1: 'Phishing (1)', 0: 'Legitimate (0)'})

# Calculate the precise count distributions
class_counts = df['Class Name'].value_counts()
print("\n--- Class Counts ---")
print(class_counts)

# ----------------------------------------------------
# 3. Generate and Plot the Graph
# ----------------------------------------------------
print("\nGenerating data distribution graphs...")

# Set up a clean, professional aesthetic style
sns.set_theme(style="whitegrid")

# Create a figure canvas with two subplots side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Plot 1: Class Distribution Bar Chart ---
sns.countplot(
    data=df, 
    x='Class Name', 
    hue='Class Name', 
    palette=['#2ecc71', '#e74c3c'], 
    ax=axes[0], 
    legend=False
)
axes[0].set_title('Class Distribution (Phishing vs Legitimate)', fontsize=14, fontweight='bold', pad=12)
axes[0].set_xlabel('URL Classification', fontsize=12)
axes[0].set_ylabel('Number of Samples', fontsize=12)

# Annotate each bar with its exact count value
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='%d', label_type='edge', padding=3, fontsize=11, fontweight='bold')

# --- Plot 2: Class Percentage Breakdown (Pie Chart) ---
axes[1].pie(
    class_counts, 
    labels=class_counts.index, 
    autopct='%1.1f%%', 
    startangle=140, 
    colors=['#2ecc71', '#e74c3c'],
    textprops={'fontsize': 12, 'fontweight': 'bold'}
)
axes[1].set_title('Dataset Ratio Split (%)', fontsize=14, fontweight='bold', pad=12)

# ----------------------------------------------------
# 4. Save Graph Layout Output
# ----------------------------------------------------
plt.tight_layout()
plt.savefig(GRAPH_OUTPUT_FILE, dpi=300)
plt.close()

print("\n==================================================")
print("GRAPH GENERATION COMPLETE!")
print(f"Your visualization has been successfully saved to:\n{GRAPH_OUTPUT_FILE}")
print("==================================================")

Dataset Loaded Successfully: 103821 rows found for plotting.

--- Class Counts ---
Class Name
Legitimate (0)    59977
Phishing (1)      43844
Name: count, dtype: int64

Generating data distribution graphs...

GRAPH GENERATION COMPLETE!
Your visualization has been successfully saved to:
C:\Users\Dypiu 539-15\Downloads\Phishing URL dataset\Phishing URL dataset\dataset_plots.png


In [4]:
import os
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# ----------------------------------------------------
# File Paths
# ----------------------------------------------------
DIR_PATH = r"C:\Users\Dypiu 539-15\Downloads\Phishing URL dataset\Phishing URL dataset"
TRAIN_INPUT = os.path.join(DIR_PATH, "train_dataset.csv")
TEST_INPUT = os.path.join(DIR_PATH, "test_dataset.csv")

# Output files with extracted features
TRAIN_OUTPUT = os.path.join(DIR_PATH, "train_features_extracted.csv")
TEST_OUTPUT = os.path.join(DIR_PATH, "test_features_extracted.csv")

# ----------------------------------------------------
# 1. Load Datasets
# ----------------------------------------------------
if not os.path.exists(TRAIN_INPUT) or not os.path.exists(TEST_INPUT):
    raise FileNotFoundError("Missing preprocessed train or test CSV file. Please run the preprocessing step first.")

train_df = pd.read_csv(TRAIN_INPUT)
test_df = pd.read_csv(TEST_INPUT)

print(f"Datasets Loaded -> Train: {len(train_df)} rows | Test: {len(test_df)} rows")

# Identify columns based on previous preprocessing output names
url_col = "URL"
norm_url_col = "normalized_url"

# Fill empty elements safely
for df in [train_df, test_df]:
    df[url_col] = df[url_col].astype(str).fillna("")
    df[norm_url_col] = df[norm_url_col].astype(str).fillna("")

# ----------------------------------------------------
# 2. Structural/Metadata Feature Extraction Function
# ----------------------------------------------------
def extract_structural_features(df, source_col):
    """
    Extracts raw URL structural markers known to correlate with phishing attempts.
    """
    # Feature 1: Raw string length of the URL
    df["feat_url_length"] = df[source_col].apply(len)
    
    # Feature 2: Count of dots (phishing links often use massive subdomains)
    df["feat_dot_count"] = df[source_col].apply(lambda x: x.count("."))
    
    # Feature 3: Count of hyphens (common in spoofed brand strings like 'secure-paypal')
    df["feat_hyphen_count"] = df[source_col].apply(lambda x: x.count("-"))
    
    # Feature 4: Contains numeric IP indicators instead of domains
    ip_pattern = r"\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}"
    df["feat_is_ip"] = df[source_col].apply(lambda x: 1 if re.search(ip_pattern, x) else 0)
    
    # Feature 5: Count of digits (phishing domains frequently generate random numbers)
    df["feat_digit_count"] = df[source_col].apply(lambda x: sum(1 for char in x if char.isdigit()))
    
    return df

print("\n[Step 1] Extracting structural metadata characteristics...")
train_df = extract_structural_features(train_df, url_col)
test_df = extract_structural_features(test_df, url_col)

# ----------------------------------------------------
# 3. TF-IDF Token Feature Extraction (NLP Engine)
# ----------------------------------------------------
print("[Step 2] Processing Text Vectorization via TF-IDF...")

# Initialize TF-IDF (keeps the top 300 most common malicious/legitimate keyword tokens)
tfidf = TfidfVectorizer(max_features=300)

# CRITICAL ML STEP: Fit & transform on training data, but ONLY transform the testing data
train_tfidf_matrix = tfidf.fit_transform(train_df[norm_url_col])
test_tfidf_matrix = tfidf.transform(test_df[norm_url_col])

# Convert sparse token matrices to DataFrames
tfidf_cols = [f"tfidf_{word}" for word in tfidf.get_feature_names_out()]

train_tfidf_df = pd.DataFrame(train_tfidf_matrix.toarray(), columns=tfidf_cols, index=train_df.index)
test_tfidf_df = pd.DataFrame(test_tfidf_matrix.toarray(), columns=tfidf_cols, index=test_df.index)

# ----------------------------------------------------
# 4. Merge Features and Save Outputs
# ----------------------------------------------------
print("[Step 3] Merging text vectors with structural metadata matrices...")

final_train_df = pd.concat([train_df, train_tfidf_df], axis=1)
final_test_df = pd.concat([test_df, test_tfidf_df], axis=1)

# Write to file system
final_train_df.to_csv(TRAIN_OUTPUT, index=False)
final_test_df.to_csv(TEST_OUTPUT, index=False)

print("\n==================================================")
print("FEATURE EXTRACTION ARCHITECTURE FINISHED!")
print(f"Total Columns Per Row: {len(final_train_df.columns)}")
print("\nProcessed matrices exported:")
print(f" -> Train Features: {TRAIN_OUTPUT}")
print(f" -> Test Features : {TEST_OUTPUT}")
print("==================================================")

Datasets Loaded -> Train: 103821 rows | Test: 25956 rows

[Step 1] Extracting structural metadata characteristics...
[Step 2] Processing Text Vectorization via TF-IDF...
[Step 3] Merging text vectors with structural metadata matrices...

FEATURE EXTRACTION ARCHITECTURE FINISHED!
Total Columns Per Row: 309

Processed matrices exported:
 -> Train Features: C:\Users\Dypiu 539-15\Downloads\Phishing URL dataset\Phishing URL dataset\train_features_extracted.csv
 -> Test Features : C:\Users\Dypiu 539-15\Downloads\Phishing URL dataset\Phishing URL dataset\test_features_extracted.csv


In [10]:
import os
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm  # Visual progress bar

# ----------------------------------------------------
# File Paths
# ----------------------------------------------------
DIR_PATH = r"C:\Users\Dypiu 539-15\Downloads\Phishing URL dataset\Phishing URL dataset"
TRAIN_INPUT = os.path.join(DIR_PATH, "train_features_extracted.csv")
TEST_INPUT = os.path.join(DIR_PATH, "test_features_extracted.csv")

TRAIN_OUTPUT = os.path.join(DIR_PATH, "train_bert_represented.csv")
TEST_OUTPUT = os.path.join(DIR_PATH, "test_bert_represented.csv")

# ----------------------------------------------------
# 1. Initialize Pre-trained AI Embeddings Engine (DistilBERT)
# ----------------------------------------------------
print("Loading pre-trained DistilBERT Architecture configurations...")
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

# Use GPU acceleration if your system has an NVIDIA card, otherwise default to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval() # Put the neural network in evaluation mode
print(f"Deep learning engine working on device: {device.type.upper()}\n")

# ----------------------------------------------------
# 2. Vector Representation Extraction Function
# ----------------------------------------------------
def extract_bert_embeddings(text_series, batch_size=64):
    """
    Transforms text sentences into dense mathematical 768-dimensional AI vectors.
    """
    texts = text_series.astype(str).tolist()
    all_embeddings = []
    
    # Process inputs in batches to keep memory usage low and safe
    for i in tqdm(range(0, len(texts), batch_size), desc="Extracting Deep Representations"):
        batch_texts = texts[i : i + batch_size]
        
        # Tokenize and pad the strings to clean matrix layouts
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=64, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad(): # Disable gradient updates to maximize speed
            outputs = model(**inputs)
            
        # Use the [CLS] token vector (the first token) as the sentence embedding representation
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_embeddings)
        
    return np.vstack(all_embeddings)

# ----------------------------------------------------
# 3. Apply Feature Representation to Data
# ----------------------------------------------------
# Load the previous feature CSV files
train_df = pd.read_csv(TRAIN_INPUT)
test_df = pd.read_csv(TEST_INPUT)

print(f"Loaded Train Dataset ({len(train_df)} rows). Processing...")
train_embeddings = extract_bert_embeddings(train_df["normalized_url"])

print(f"\nLoaded Test Dataset ({len(test_df)} rows). Processing...")
test_embeddings = extract_bert_embeddings(test_df["normalized_url"])

# ----------------------------------------------------
# 4. Map, Flatten and Save Structural Matrices
# ----------------------------------------------------
print("\nMapping array tensors into tabular spreadsheet tables...")
# Generate unique names for all 768 neural dimensions (bert_0, bert_1, ... bert_767)
bert_cols = [f"bert_{i}" for i in range(train_embeddings.shape[1])]

train_bert_df = pd.DataFrame(train_embeddings, columns=bert_cols, index=train_df.index)
test_bert_df = pd.DataFrame(test_embeddings, columns=bert_cols, index=test_df.index)

# Combine the original features, tfidf scores, and new BERT embeddings side-by-side
final_train = pd.concat([train_df, train_bert_df], axis=1)
final_test = pd.concat([test_df, test_bert_df], axis=1)

# Write out to new csv outputs
final_train.to_csv(TRAIN_OUTPUT, index=False)
final_test.to_csv(TEST_OUTPUT, index=False)

print("\n==================================================")
print("FEATURE REPRESENTATION REFACTOR COMPLETE!")
print(f"Saved Output -> Train Vector Space: {TRAIN_OUTPUT}")
print(f"Saved Output -> Test Vector Space : {TEST_OUTPUT}")
print("==================================================")

Loading pre-trained DistilBERT Architecture configurations...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

C:\Users\Dypiu 539-15\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dypiu 539-15\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

Deep learning engine working on device: CPU

Loaded Train Dataset (103821 rows). Processing...


Extracting Deep Representations: 100%|█████████████████████████████████████████████| 1623/1623 [28:10<00:00,  1.04s/it]



Loaded Test Dataset (25956 rows). Processing...


Extracting Deep Representations: 100%|███████████████████████████████████████████████| 406/406 [07:09<00:00,  1.06s/it]



Mapping array tensors into tabular spreadsheet tables...

FEATURE REPRESENTATION REFACTOR COMPLETE!
Saved Output -> Train Vector Space: C:\Users\Dypiu 539-15\Downloads\Phishing URL dataset\Phishing URL dataset\train_bert_represented.csv
Saved Output -> Test Vector Space : C:\Users\Dypiu 539-15\Downloads\Phishing URL dataset\Phishing URL dataset\test_bert_represented.csv


In [7]:
pip install torch

   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
    --------------------------------------- 2.6/123.0 MB 16.7 MB/s eta 0:00:08
   - -------------------------------------- 5.5/123.0 MB 15.4 MB/s eta 0:00:08
   -- ------------------------------------- 8.1/123.0 MB 14.8 MB/s eta 0:00:08
   --- ------------------------------------ 10.0/123.0 MB 13.4 MB/s eta 0:00:09
   --- ------------------------------------ 11.5/123.0 MB 11.9 MB/s eta 0:00:10
   ---- ----------------------------------- 13.1/123.0 MB 11.0 MB/s eta 0:00:10
   ---- ----------------------------------- 14.4/123.0 MB 10.4 MB/s eta 0:00:11
   ----- ---------------------------------- 16.5/123.0 MB 10.4 MB/s eta 0:00:11
   ------ --------------------------------- 18.6/123.0 MB 10.3 MB/s eta 0:00:11
   ------ --------------------------------- 21.0/123.0 MB 10.4 MB/s eta 0:00:10
   ------- -------------------------------- 23.9/123.0 MB 10.8 MB/s eta 0:00:10
   -------- ------------------------------- 27.0/123

In [9]:
pip install transformers

   ---------------------------------------- 0.0/11.5 MB ? eta -:--:--
   ---------------------------- ----------- 8.1/11.5 MB 40.6 MB/s eta 0:00:01
   ---------------------------------------- 11.5/11.5 MB 40.0 MB/s  0:00:00
   ---------------------------------------- 0.0/765.1 kB ? eta -:--:--
   ---------------------------------------- 765.1/765.1 kB 28.5 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 4.0/4.0 MB 45.2 MB/s  0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 32.0 MB/s  0:00:00

  Attempting uninstall: regex

    Found existing installation: regex 2025.9.1

    Uninstalling regex-2025.9.1:

      Successfully uninstalled regex-2025.9.1

   ----- ---------------------------------- 1/7 [regex]
  Attempting uninstall: click
   ----- ---------------------------------- 1/7 [regex]
    Found existing installation: clic